# 02. Agent Execution Loop

This notebook builds a rigorously typed, bounded execution loop that demonstrates budgets, error classification, retry backoffs, no-progress detection, dynamic replanning, and graph mapping.

In [1]:
import json
import time
import os
from typing import Dict, Any, List, Optional, Literal, Set
from pydantic import BaseModel, Field, ValidationError

print('Environment initialized.')

Environment initialized.


## Part 1: Naive Agent Loop

An unbounded loop with no step limit and no error classification is dangerous.

In [2]:
def naive_decide(state):
    if 'error' in str(state.get('last_obs')).lower(): return 'retry_same_action'
    return 'final_answer'

state = {'last_obs': 'API Error: Timeout'}
steps = 0
print('Starting naive loop...')
while state.get('last_obs') != 'success':
    action = naive_decide(state)
    state['last_obs'] = 'API Error: Timeout' # simulate recurring failure
    steps += 1
    if steps > 5:
        print('DANGER: Runaway loop detected. Forced exit.')
        break

Starting naive loop...
DANGER: Runaway loop detected. Forced exit.


## Part 2: Typed State & Actions

To build a professional loop, we must strongly type the agent's state, tracking evidence, budgets, hypothesis, and explicit decisions.

In [3]:
class AgentState(BaseModel):
    goal: str
    scenario_type: str = 'happy_path'
    evidence: List[str] = Field(default_factory=list)
    steps: int = 0
    tool_calls: int = 0
    retries: int = 0
    replans: int = 0
    seen_actions: Set[str] = Field(default_factory=set)
    terminal_reason: Optional[str] = None
    current_hypothesis: Optional[str] = None
    replan_reason: Optional[str] = None
    history: List[Dict[str, Any]] = Field(default_factory=list)

class ToolCall(BaseModel):
    # Only legitimate Northstar tools are permitted here.
    tool: Literal['get_service_health', 'query_checkout_logs', 'search_incidents', 'get_runbook', 'get_recent_deployments']
    arguments: Dict[str, Any]

class AgentDecision(BaseModel):
    thought: str
    tool_call: Optional[ToolCall] = None
    final_answer: Optional[str] = None
    replan: bool = False

## Part 3: Explicit Tool Registry

Tools are strictly registered with typed argument schemas and retry policies. We use specific inputs to trigger transient and fatal errors, avoiding synthetic 'fail' tools.

In [4]:
class RegionQuery(BaseModel): region: str
class ServiceQuery(BaseModel): service: str
class TopicQuery(BaseModel): topic: str

def get_service_health(args: RegionQuery) -> str:
    if args.region == 'FAIL_EU': raise TimeoutError('HTTP 504 Gateway Timeout')
    if args.region == 'EU': return 'Status: Degraded (Checkout DB High Latency)'
    return 'Status: Healthy'

def query_checkout_logs(args: RegionQuery) -> str:
    if args.region == 'DENY_EU': raise PermissionError('Permission Denied: write access required or unauthorized read')
    if args.region == 'EU': return '[EU] 45 timeouts matching payment gateway.'
    return 'No logs found.'

def search_incidents(args: ServiceQuery) -> str:
    return 'No active incident for ' + args.service

def get_runbook(args: TopicQuery) -> str:
    if 'timeout' in args.topic.lower(): return 'Runbook: Check recent deployments. If recent deploy, consider rollback.'
    return 'Runbook not found.'

def get_recent_deployments(args: ServiceQuery) -> str:
    if args.service == 'checkout': return 'Deployment v1.14 shipped 20 minutes ago to checkout.'
    return 'No recent deployments.'

tool_registry = {
    'get_service_health': {'func': get_service_health, 'schema': RegionQuery, 'read_only': True, 'max_retries': 3},
    'query_checkout_logs': {'func': query_checkout_logs, 'schema': RegionQuery, 'read_only': True, 'max_retries': 3},
    'search_incidents': {'func': search_incidents, 'schema': ServiceQuery, 'read_only': True, 'max_retries': 3},
    'get_runbook': {'func': get_runbook, 'schema': TopicQuery, 'read_only': True, 'max_retries': 3},
    'get_recent_deployments': {'func': get_recent_deployments, 'schema': ServiceQuery, 'read_only': True, 'max_retries': 3}
}
print('Explicit Tool Registry Initialized.')

Explicit Tool Registry Initialized.


## Part 4: The Bounded Execution Loop

The runtime catches errors, classifies them (Transient vs Fatal), tracks steps (all attempts) vs tool_calls (successful dispatches), and explicitly detects looping (no-progress) by comparing repeated state or repeated evidence.

In [5]:
def dispatch_tool(tool_name: str, args: dict) -> str:
    entry = tool_registry[tool_name]
    try:
        val_args = entry['schema'](**args)
        return str(entry['func'](val_args))
    except ValidationError as e: return f'ValidationError: {e.errors()[0]["msg"]}'
    except TimeoutError as e: return f'TransientError: {str(e)}'
    except PermissionError as e: return f'FatalError: {str(e)}'
    except Exception as e: return f'UnknownError: {str(e)}'

def run_bounded_loop(state: AgentState, decision_model, max_steps=8):
    print(f"\n[Runtime Started] Goal: {state.goal}")
    last_obs = None
    repeated_obs_count = 0

    while state.steps < max_steps:
        state.steps += 1
        decision: AgentDecision = decision_model(state)
        state.history.append({'role': 'model', 'decision': decision})
        print(f"\nStep {state.steps} | Thought: {decision.thought}")
        if state.current_hypothesis: print(f"Hypothesis: {state.current_hypothesis}")

        if decision.final_answer:
            print(f"[Terminal] Final Answer: {decision.final_answer}")
            state.terminal_reason = 'SUCCESS'
            break

        if decision.tool_call:
            t_name = decision.tool_call.tool
            args = decision.tool_call.arguments
            fingerprint = f"{t_name}:{args}"

            obs = dispatch_tool(t_name, args)
            print(f"[Observation] {obs}")

            if 'FatalError' in obs or 'ValidationError' in obs:
                print("FATAL ERROR: Policy block or invalid arg. Terminating.")
                state.terminal_reason = 'POLICY_BLOCK'
                break

            if 'TransientError' in obs:
                print("TRANSIENT ERROR: Simulating backoff and retrying...")
                if state.retries < tool_registry[t_name]['max_retries']:
                    state.retries += 1
                    continue
                else:
                    print("Max retries exceeded.")
                    state.terminal_reason = 'TOOL_FAILURE'
                    break

            state.tool_calls += 1
            state.seen_actions.add(fingerprint)
            state.evidence.append(obs)

            # No-progress detection: repeatedly seeing the exact same observation without a state change.
            if obs == last_obs:
                repeated_obs_count += 1
            else:
                repeated_obs_count = 0
            last_obs = obs

            if repeated_obs_count > 1:
                print("NO_PROGRESS: Repeated identical evidence with no change in path. Escalating.")
                state.terminal_reason = 'NO_PROGRESS'
                break

    if not state.terminal_reason:
        state.terminal_reason = 'STEP_BUDGET_EXHAUSTED'
    print(f"Loop terminated with reason: {state.terminal_reason}")
    return state

## Part 5: Deterministic Decision Model

This mock model illustrates how an agent changes its hypothesis when new evidence invalidates assumptions, triggering a dynamic replan.

In [6]:
def mock_model(state: AgentState) -> AgentDecision:
    scen = state.scenario_type

    if scen == 'happy_path':
        if state.tool_calls == 0: return AgentDecision(thought="Check health.", tool_call=ToolCall(tool='get_service_health', arguments={'region': 'EU'}))
        if state.tool_calls == 1: return AgentDecision(thought="Check logs.", tool_call=ToolCall(tool='query_checkout_logs', arguments={'region': 'EU'}))
        return AgentDecision(thought="Sufficient evidence gathered.", final_answer="Degraded EU checkout due to gateway timeouts.")

    if scen == 'replan':
        if state.tool_calls == 0:
            state.current_hypothesis = "Network outage"
            return AgentDecision(thought="Assume network. Check logs.", tool_call=ToolCall(tool='query_checkout_logs', arguments={'region': 'EU'}))
        if state.tool_calls == 1:
            return AgentDecision(thought="Timeouts found. Getting runbook.", tool_call=ToolCall(tool='get_runbook', arguments={'topic': 'timeout'}))
        if state.tool_calls == 2:
            return AgentDecision(thought="Runbook says check deployments.", tool_call=ToolCall(tool='get_recent_deployments', arguments={'service': 'checkout'}))
        if state.tool_calls == 3:
            # New evidence contradicts network outage hypothesis.
            state.current_hypothesis = "Bad deployment v1.14 caused timeouts"
            state.replan_reason = "Recent deployment found, contradicting network outage."
            state.replans += 1
            print(f"\n>>> REPLAN TRIGGERED: {state.replan_reason} <<<")
            return AgentDecision(thought="Hypothesis changed. Recommending rollback.", final_answer="Recommend rollback of v1.14.")

    if scen == 'repeated_action':
        return AgentDecision(thought="I will just keep checking logs.", tool_call=ToolCall(tool='query_checkout_logs', arguments={'region': 'EU'}))

    if scen == 'transient_timeout':
        # Triggers a TransientError, loop retries until TOOL_FAILURE
        return AgentDecision(thought="Checking FAIL_EU.", tool_call=ToolCall(tool='get_service_health', arguments={'region': 'FAIL_EU'}))

    if scen == 'permission_denied':
        # Triggers FatalError
        return AgentDecision(thought="Checking DENY_EU logs.", tool_call=ToolCall(tool='query_checkout_logs', arguments={'region': 'DENY_EU'}))

    if scen == 'insufficient_evidence':
        if state.tool_calls == 0: return AgentDecision(thought="Check US health.", tool_call=ToolCall(tool='get_service_health', arguments={'region': 'US'}))
        return AgentDecision(thought="I have no idea what is wrong.", final_answer="I don't know what's wrong.")

    return AgentDecision(thought="Unknown", final_answer="Abort")

## Part 6: Evaluation Harness

We evaluate 6 unique scenarios to ensure the bounding, budget, and retry logic behaves identically and predictably across trajectories.

In [7]:
scenarios = [
    ('happy_path', 'happy_path'),
    ('transient_timeout', 'transient_timeout'),
    ('permission_denied', 'permission_denied'),
    ('repeated_action', 'repeated_action'),
    ('changed_deployment_replan', 'replan'),
    ('insufficient_evidence', 'insufficient_evidence')
]
results = []
for name, stype in scenarios:
    print(f"\n--- Evaluating Scenario: {name} ---")
    st = AgentState(goal="Diagnose incident", scenario_type=stype)
    final_st = run_bounded_loop(st, mock_model, max_steps=8)
    results.append({
        'scenario': name,
        'terminal_reason': final_st.terminal_reason,
        'steps': final_st.steps,
        'tool_calls': final_st.tool_calls,
        'retries': final_st.retries,
        'replans': final_st.replans,
        'violations': 1 if final_st.terminal_reason == 'POLICY_BLOCK' else 0
    })

print("\n--- EVALUATION METRICS ---")
import pandas as pd
display(pd.DataFrame(results))


--- Evaluating Scenario: happy_path ---

[Runtime Started] Goal: Diagnose incident

Step 1 | Thought: Check health.
[Observation] Status: Degraded (Checkout DB High Latency)

Step 2 | Thought: Check logs.
[Observation] [EU] 45 timeouts matching payment gateway.

Step 3 | Thought: Sufficient evidence gathered.
[Terminal] Final Answer: Degraded EU checkout due to gateway timeouts.
Loop terminated with reason: SUCCESS

--- Evaluating Scenario: transient_timeout ---

[Runtime Started] Goal: Diagnose incident

Step 1 | Thought: Checking FAIL_EU.
[Observation] TransientError: HTTP 504 Gateway Timeout
TRANSIENT ERROR: Simulating backoff and retrying...

Step 2 | Thought: Checking FAIL_EU.
[Observation] TransientError: HTTP 504 Gateway Timeout
TRANSIENT ERROR: Simulating backoff and retrying...

Step 3 | Thought: Checking FAIL_EU.
[Observation] TransientError: HTTP 504 Gateway Timeout
TRANSIENT ERROR: Simulating backoff and retrying...

Step 4 | Thought: Checking FAIL_EU.
[Observation] Transi

,scenario,terminal_reason,steps,tool_calls,retries,replans,violations
0,happy_path,SUCCESS,3,2,0,0,0
1,transient_timeout,TOOL_FAILURE,4,0,3,0,0
2,permission_denied,POLICY_BLOCK,1,0,0,0,1
3,repeated_action,NO_PROGRESS,3,3,0,0,0
4,changed_deployment_replan,SUCCESS,4,3,0,1,0
5,insufficient_evidence,SUCCESS,2,1,0,0,0


## Part 7: Reflection and Plan-and-Execute

Agents can evaluate their own intermediate steps using Reflection and revise coarse plans as evidence invalidates initial assumptions.

In [8]:
print("--- Reflection Rubric ---")
def evaluate_reflection_rubric(draft: str, evidence: list[str]) -> bool:
    print(f"Reflecting on draft: '{draft}'")
    if "rollback" in draft and not any("Deployment" in e for e in evidence):
        print("-> Rubric Failed: Proposes rollback without evidence of deployment.")
        return False
    if "restart db" in draft:
        print("-> Rubric Failed: Unsupported production action.")
        return False
    print("-> Rubric Passed: Safe and grounded.")
    return True

evaluate_reflection_rubric("We should restart db and rollback.", evidence=[])
evaluate_reflection_rubric("Recommend rollback of v1.14.", evidence=['Deployment v1.14 shipped'])

print("\n--- Plan-and-Execute Concept ---")
print("Initial Plan: [check_health, check_logs, rollback_db]")
print("Step 1: check_health returns 'All Systems Operational'.")
print("Result: Evidence invalidates need for checkout logs. Plan revised dynamically to: [search_recent_incidents_instead]")

--- Reflection Rubric ---
Reflecting on draft: 'We should restart db and rollback.'
-> Rubric Failed: Proposes rollback without evidence of deployment.
Reflecting on draft: 'Recommend rollback of v1.14.'
-> Rubric Passed: Safe and grounded.

--- Plan-and-Execute Concept ---
Initial Plan: [check_health, check_logs, rollback_db]
Step 1: check_health returns 'All Systems Operational'.
Result: Evidence invalidates need for checkout logs. Plan revised dynamically to: [search_recent_incidents_instead]


## Part 8: Event-Driven Resume (Durable Execution)

For production reliability, workflows track process state and process events idempotently using explicit Event and Workflow IDs.

In [9]:
class WorkflowState:
    def __init__(self, workflow_id):
        self.workflow_id = workflow_id
        self.processed_events = set()
        self.resumed_state = {}

db = {}
def process_event(workflow_id: str, event_id: str, data: str):
    if workflow_id not in db: db[workflow_id] = WorkflowState(workflow_id)
    wf = db[workflow_id]
    if event_id in wf.processed_events:
        print(f"Idempotency Guard: Event {event_id} already processed. Skipping.")
        return
    print(f"Processing new event {event_id}: {data}")
    wf.processed_events.add(event_id)
    wf.resumed_state['last_event'] = data

process_event("WF-100", "EVT-1", "Webhook: Service down")
process_event("WF-100", "EVT-1", "Webhook: Service down (Duplicate delivery)")

Processing new event EVT-1: Webhook: Service down
Idempotency Guard: Event EVT-1 already processed. Skipping.


## Part 9: LangGraph Mapping

Instead of manually writing `while` loops and condition checks, modern state-machine orchestration like LangGraph models these explicitly as nodes and edges.

In [10]:
try:
    from langgraph.graph import StateGraph, END
    from typing import TypedDict

    class GraphState(TypedDict):
        goal: str
        tool_calls: int
        terminal_reason: str

    def decide_node(state: GraphState):
        print(f"[LangGraph Node: Decide] tool_calls={state['tool_calls']}")
        return {'tool_calls': state['tool_calls'] + 1}

    def tool_node(state: GraphState):
        print("[LangGraph Node: Tool] executing tool...")
        return {}

    def evaluate_evidence(state: GraphState):
        print("[LangGraph Node: Evaluate] checking evidence...")
        if state['tool_calls'] >= 2:
            return {'terminal_reason': 'SUCCESS'}
        return {}

    def should_continue(state: GraphState) -> str:
        if state.get('terminal_reason') == 'SUCCESS':
            return 'recommend'
        return 'tool'

    def recommend(state: GraphState):
        print("[LangGraph Node: Recommend] Finalizing output.")
        return {}

    workflow = StateGraph(GraphState)
    workflow.add_node('decide', decide_node)
    workflow.add_node('tool', tool_node)
    workflow.add_node('evaluate', evaluate_evidence)
    workflow.add_node('recommend', recommend)

    workflow.set_entry_point('decide')
    workflow.add_edge('decide', 'evaluate')
    workflow.add_conditional_edges('evaluate', should_continue, {'tool': 'tool', 'recommend': 'recommend'})
    workflow.add_edge('tool', 'decide')
    workflow.add_edge('recommend', END)

    app = workflow.compile()
    print("\n--- LangGraph Compiled. Running Northstar Scenario ---")
    app.invoke({'goal': 'Investigate EU', 'tool_calls': 0, 'terminal_reason': ''})
except ImportError:
    print("LangGraph not installed. See code structure above.")


--- LangGraph Compiled. Running Northstar Scenario ---
[LangGraph Node: Decide] tool_calls=0
[LangGraph Node: Evaluate] checking evidence...
[LangGraph Node: Tool] executing tool...
[LangGraph Node: Decide] tool_calls=1
[LangGraph Node: Evaluate] checking evidence...
[LangGraph Node: Recommend] Finalizing output.


## Part 10: Optional Real LLM (OpenAI)

Uses OpenAI API if `OPENAI_API_KEY` is present. It runs inside the **same exact boundary runtime**.

In [11]:
api_key = os.getenv('OPENAI_API_KEY')
if not api_key:
    print("No OPENAI_API_KEY found. Skipping real LLM call.")
else:
    from openai import OpenAI
    client = OpenAI(api_key=api_key)

    openai_tools = [
        {'type': 'function', 'function': {'name': 'get_service_health', 'description': 'Check region health', 'parameters': {'type': 'object', 'properties': {'region': {'type': 'string'}}, 'required': ['region']}}},
        {'type': 'function', 'function': {'name': 'query_checkout_logs', 'description': 'Check logs', 'parameters': {'type': 'object', 'properties': {'region': {'type': 'string'}}, 'required': ['region']}}}
    ]

    def openai_decision(state: AgentState) -> AgentDecision:
        messages = [{'role': 'system', 'content': 'You are a diagnostic assistant. Use tools.'}]
        messages.append({'role': 'user', 'content': state.goal})
        for h in state.history:
            if h['role'] == 'model' and h['decision'].tool_call:
                messages.append({'role': 'assistant', 'content': h['decision'].thought, 'tool_calls': [{'id': 'c1', 'type': 'function', 'function': {'name': h['decision'].tool_call.tool, 'arguments': json.dumps(h['decision'].tool_call.arguments)}}]})
            elif h['role'] == 'environment':
                messages.append({'role': 'tool', 'tool_call_id': 'c1', 'name': h['tool'], 'content': h['observation']})

        response = client.chat.completions.create(model='gpt-4o-mini', messages=messages, tools=openai_tools)
        msg = response.choices[0].message

        if msg.tool_calls:
            tc = msg.tool_calls[0].function
            if tc.name in tool_registry:
                return AgentDecision(thought=msg.content or "Tool call", tool_call=ToolCall(tool=tc.name, arguments=json.loads(tc.arguments)))
            else:
                return AgentDecision(thought="Invalid tool", final_answer="Tried invalid tool.")
        else:
            return AgentDecision(thought="Final answer ready.", final_answer=msg.content)

    print("\n--- Running Real OpenAI Model in Bounded Loop ---")
    run_bounded_loop(AgentState(goal="Check EU health"), openai_decision, max_steps=5)

No OPENAI_API_KEY found. Skipping real LLM call.
